# Teste da Ferramenta de Enriquecimento de Dados

Este notebook demonstra e testa a funcionalidade `data_enricher` implementada na Fase 2 (Tratamento).

**Objetivos:**
1. Verificar se o processo falha corretamente quando a chave de busca (`lookup_key`) contém valores duplicados.
2. Verificar se o processo é executado com sucesso com dados válidos e se o arquivo de saída é gerado corretamente.

In [4]:
import sys
import os
import pandas as pd

# --- Configuração de Path Robusta ---
# Encontra o diretório raiz do projeto dinamicamente, procurando pela pasta 'src'.
# Isso torna o notebook funcional independentemente de onde o kernel do Jupyter é iniciado.
path = os.path.abspath('.')
while 'src' not in os.listdir(path):
    path = os.path.dirname(path)
    if path == os.path.dirname(path): # Chegou na raiz do sistema de arquivos
        raise FileNotFoundError("Não foi possível encontrar o diretório 'src'. Verifique se o notebook está dentro do projeto.")

project_root = path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Raiz do projeto encontrada: {project_root}")

# Agora a importação deve funcionar
from src.main.orchestrator import main

print("Módulo 'main' importado com sucesso!")

Raiz do projeto encontrada: d:\w\analise-dados
Módulo 'main' importado com sucesso!


## Cenário 1: Teste de Falha com Chave Duplicada

Neste cenário, usamos o arquivo `agencias.csv` que contém uma `Sigla` duplicada ("ANAC"). A ferramenta de enriquecimento deve identificar essa duplicata e lançar uma exceção, interrompendo o processo.

In [ ]:
# Armazena os argumentos originais da linha de comando
original_argv = sys.argv

# Define os argumentos para simular a execução via linha de comando
# Apontamos para o arquivo de configuração que usa o CSV com a sigla duplicada.
sys.argv = [
    'src/main/orchestrator.py',
    '-d', 'data/sample',
    '--phase', 'treatment',
    '--enrich-data', 'data/test_enricher/config_enricher.json'
]

print(f"Executando com os argumentos: {sys.argv}")

try:
    main()
except Exception as e:
    print("\n--- CAPTURA DE EXCEÇÃO ESPERADA ---")
    print(f"A execução falhou como esperado. Mensagem: {e}")
    print("------------------------------------")
finally:
    # Restaura os argumentos originais
    sys.argv = original_argv

# Verifica se o arquivo de saída NÃO foi criado
output_path = 'D:/w/analise-dados/data/test_enricher/beneficios_enriquecido.csv'
if not os.path.exists(output_path):
    print(f"\nSUCESSO: O arquivo de saída '{output_path}' não foi criado, como esperado.")
else:
    print(f"\nFALHA: O arquivo de saída '{output_path}' foi criado indevidamente.")

Executando com os argumentos: ['src/main/orchestrator.py', '-d', 'data/sample', '--phase', 'treatment', '--enrich-data', 'data/test_enricher/config_enricher.json']


usage: orchestrator.py [-h] -d DATA_PROJECT_PATH -p
                       {discovery,treatment,exploratory,visualization}
                       [-o {text,interactive}] [--compare-fields]
orchestrator.py: error: unrecognized arguments: --enrich-data data/test_enricher/config_enricher.json


SystemExit: 2

## Cenário 2: Teste de Sucesso

Agora, vamos corrigir os dados de teste e executar o processo novamente.

1. Criamos um novo arquivo `agencias_corrigido.csv` sem a `Sigla` duplicada.
2. Criamos um novo arquivo de configuração `config_enricher_sucesso.json` para usar o arquivo corrigido.
3. Executamos a ferramenta.
4. Verificamos se o arquivo `beneficios_enriquecido.csv` foi criado e se o conteúdo está correto.

In [ ]:
# Conteúdo do CSV corrigido (sem a sigla 'ANAC' duplicada)
agencias_corrigido_content = ''''ID,Sigla,Nome
1,ANEEL,Agência Nacional de Energia Elétrica
2,ANATEL,Agência Nacional de Telecomunicações
3,ANVISA,Agência Nacional de Vigilância Sanitária
4,ANAC,Agência Nacional de Aviação Civil
5,ANTAQ,Agência Nacional de Transportes Aquaviários
'''

# Conteúdo do JSON de configuração para o teste de sucesso
config_sucesso_content = ''''{{
    "main_file": "data/test_enricher/beneficios.csv",
    "main_key": "Sigla_Agencia",
    "lookup_file": "data/test_enricher/agencias_corrigido.csv",
    "lookup_key": "Sigla",
    "columns_to_add": ["ID"],
    "output_file": "data/test_enricher/beneficios_enriquecido.csv"
}}'''

# Caminhos dos novos arquivos
agencias_corrigido_path = 'D:/w/analise-dados/data/test_enricher/agencias_corrigido.csv'
config_sucesso_path = 'D:/w/analise-dados/data/test_enricher/config_enricher_sucesso.json'

# Escreve os novos arquivos
with open(agencias_corrigido_path, 'w', encoding='utf-8') as f:
    f.write(agencias_corrigido_content)

with open(config_sucesso_path, 'w', encoding='utf-8') as f:
    f.write(config_sucesso_content)

print(f"Arquivo corrigido criado em: {agencias_corrigido_path}")
print(f"Arquivo de configuração de sucesso criado em: {config_sucesso_path}")

In [ ]:
# Armazena os argumentos originais da linha de comando
original_argv = sys.argv

# Define os argumentos para a execução de sucesso
sys.argv = [
    'src/main/orchestrator.py',
    '--phase', 'treatment',
    '--enrich-data', 'data/test_enricher/config_enricher_sucesso.json'
]

print(f"Executando com os argumentos: {sys.argv}")

try:
    main()
    print("\nSUCESSO: O processo de enriquecimento foi concluído sem erros.")
except Exception as e:
    print(f"\nFALHA: A execução falhou inesperadamente. Mensagem: {e}")
finally:
    # Restaura os argumentos originais
    sys.argv = original_argv

In [ ]:
# Verifica se o arquivo de saída foi criado e exibe seu conteúdo
output_path = 'D:/w/analise-dados/data/test_enricher/beneficios_enriquecido.csv'

if os.path.exists(output_path):
    print(f"\nO arquivo de saída '{output_path}' foi criado com sucesso.")
    print("Conteúdo do arquivo enriquecido:")
    df_resultado = pd.read_csv(output_path)
    display(df_resultado)
else:
    print(f"\nFALHA: O arquivo de saída '{output_path}' não foi encontrado.")